# Time series momentum: Evidence from the European equity market

**Authors:** Darko Vuković, Salvatore Ingenito, Moinak Maiti

**Published:** 2023-01-01

**URL:** [https://doi.org/10.1016/j.heliyon.2023.e12989](https://doi.org/10.1016/j.heliyon.2023.e12989)

**Abstract:**
This study empirically analyzes time series momentum (TSM) in the European equity market between 2000 & 2020. The study produces additional evidence on TSM where a significant and persistent market price anomaly enables investors to earn abnormal returns. To achieve this goal the present study implements a pooled autoregressive model to test the predictability power of European equity indices of future returns. The results indicate that strategies based on TSM are in line with the discussed literature and enable market agents to earn returns above the market (0.71% per month) by using a six-factor model.

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## Phase 1: Configuration

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

# Configuration
symbol = '^STOXX50E'  # EURO STOXX 50 Index
start_date = '2000-01-01'
end_date = '2020-12-31'
lookback_period = 252  # 1 year lookback for momentum
risk_free_rate = 0.01  # 1% annual risk-free rate

## Phase 2: Data Download and Feature Engineering

In [ ]:
data = yf.download(symbol, start=start_date, end=end_date)
data['Returns'] = data['Adj Close'].pct_change()
data['Momentum'] = data['Adj Close'].pct_change(lookback_period)

## Phase 3: Signal Generation and Portfolio Construction

In [ ]:
data['Signal'] = np.where(data['Momentum'] > 0, 1, -1)
data['Strategy_Returns'] = data['Signal'].shift(1) * data['Returns']

## Phase 4: Vectorized Backtest

In [ ]:
data['Cumulative_Strategy_Returns'] = (1 + data['Strategy_Returns']).cumprod()

## Phase 5: Performance Metrics

In [ ]:
def calculate_performance(data):
    strategy_returns = data['Strategy_Returns'].dropna()
    sharpe_ratio = (strategy_returns.mean() - risk_free_rate/252) / strategy_returns.std() * np.sqrt(252)
    downside_returns = strategy_returns[strategy_returns < 0]
    sortino_ratio = (strategy_returns.mean() - risk_free_rate/252) / downside_returns.std() * np.sqrt(252)
    max_drawdown = (data['Cumulative_Strategy_Returns'].cummax() - data['Cumulative_Strategy_Returns']).max()
    calmar_ratio = (strategy_returns.mean() * 252) / max_drawdown
    return sharpe_ratio, sortino_ratio, calmar_ratio, max_drawdown

sharpe, sortino, calmar, max_dd = calculate_performance(data)

print(f"Sharpe Ratio: {sharpe:.2f}")
print(f"Sortino Ratio: {sortino:.2f}")
print(f"Calmar Ratio: {calmar:.2f}")
print(f"Max Drawdown: {max_dd:.2%}")

# Plot equity curve
data['Cumulative_Strategy_Returns'].plot(title='Cumulative Strategy Returns')
plt.show()

## Phase 6: Monitoring Stub

In [ ]:
# Placeholder for monitoring code
# In practice, this would involve setting up alerts or dashboards to monitor live performance
pass